#1. LLMs and Chat Models
Focus: Handling structured message types (System, Human, AI).

In [10]:
!pip install -q "langchain==0.2.16" "langchain-core==0.2.43" "langchain-groq"

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

# Initialize the model
model = ChatOpenAI(model="gpt-4o", temperature=0)

# Define a structured conversation
messages = [
    SystemMessage(content="You are a helpful Data Science tutor."),
    HumanMessage(content="What is the difference between an LLM and a Chat Model?")
]

# Execute
response = model.invoke(messages)
print(response.content)

#2. Prompts and Prompt Templates
Focus: Dynamic variable injection and reusable blueprints.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Create a template with a dynamic variable {topic}
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are an expert technical writer."),
    ("user", "Write a short summary about {topic}.")
])

# Generate a specific prompt
final_prompt = prompt_template.invoke({"topic": "Vector Embeddings"})
print(final_prompt)

#3. Chains (Modern LCEL)
Focus: Piping components together using the | operator.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# Build the pipeline: Prompt -> LLM -> String Parser
chain = prompt_template | model | StrOutputParser()

# Run the chain
result = chain.invoke({"topic": "LangChain Expression Language"})
print(result)

#4. Memory
Focus: Managing conversation state in a stateless API environment.

In [ ]:
from langchain.memory import ChatMessageHistory

# Initialize history
history = ChatMessageHistory()

# Simulating a conversation flow
history.add_user_message("Hi, my name is Nandish.")
history.add_ai_message("Hello Nandish! How can I help you today?")

# Retrieving stored messages for the next prompt
print(history.messages)

#5. Document Loaders
Focus: Importing external unstructured data.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Load a local PDF file
loader = PyPDFLoader("internship_report.pdf")
pages = loader.load_and_split()

# Preview the first page content
print(pages[0].page_content)

#6. Indexes and Vector Stores
Focus: Turning text into searchable mathematical vectors (Embeddings).


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Convert documents into a searchable database
vector_db = Chroma.from_documents(
    documents=pages,
    embedding=OpenAIEmbeddings(),
    persist_directory="./my_vector_db"
)

# Search for the most relevant chunk
docs = vector_db.similarity_search("What are the internship goals?")
print(docs[0].page_content)

#7. Tools
Focus: Creating functional capabilities the LLM can call.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults

# Initialize a search tool (requires TAVILY_API_KEY)
search_tool = TavilySearchResults(max_results=2)

# Manually invoking a tool
results = search_tool.invoke({"query": "LangChain updates 2026"})
print(results)

#8. Agents
Focus: The "Reasoning Loop" that decides which tool to use.

In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_openai_functions_agent

# 1. Define tools in a list
tools = [search_tool]

# 2. Pull a prompt from the community hub
prompt = hub.pull("hwchase17/openai-functions-agent")

# 3. Initialize the agent logic
agent = create_openai_functions_agent(model, tools, prompt)

# 4. Create the executor (The 'engine')
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 5. Run the agent
agent_executor.invoke({"input": "Search for the latest trends in GenAI for 2026."})